In [ ]:
# -*- coding: utf-8 -*-

Tarea3IA.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1cAioDd2ttuW0VVjYS_Tc91YGnhYgGvpy

# Tarea Nº 3 - Inteligencia Artificial
### Alumno: Lucino Zuñiga
### Alumno: Javiera Manzo González
### Fecha de entrega: 26 de Noviembre 2024

##Introducción:

##Item I:
Reproduzca el tutorial implementando la arquitectura. Comente paso a paso lo que se hace en sus propias palabras.

In [ ]:
import numpy as np

class Neuron:

    def __init__(self, n_in, n_weights_per_edge, weights_range=None):
        self.n_in = n_in
        self.n_weights_per_edge = n_weights_per_edge
        weights_range = [-1, 1] if weights_range is None else weights_range
        self.weights = np.random.uniform(weights_range[0], weights_range[-1], size=(self.n_in, self.n_weights_per_edge))
        self.bias = 0
        self.xin = None
        self.xmid = None
        self.xout = None
        self.dxout_dxmid = None
        self.dxout_dbias = None
        self.dxmid_dw = None
        self.dxmid_dxin = None
        self.dxout_dxin = None
        self.dxout_dw = None
        self.dloss_dw = np.zeros((self.n_in, self.n_weights_per_edge))
        self.dloss_dbias = 0

    def __call__(self, xin):
        self.xin = np.array(xin)
        self.get_xmid()
        self.get_xout()

        self.get_dxout_dxmid()
        self.get_dxout_dbias()
        self.get_dxmid_dw()
        self.get_dxmid_dxin()

        assert self.dxout_dxmid.shape == (self.n_in, )
        assert self.dxmid_dxin.shape == (self.n_in, )
        assert self.dxmid_dw.shape == (self.n_in, self.n_weights_per_edge)

        self.get_dxout_dxin()
        self.get_dxout_dw()

        return self.xout

    def get_xmid(self):
        pass

    def get_xout(self):
        pass

    def get_dxout_dxmid(self):
        pass

    def get_dxout_dbias(self):
        pass

    def get_dxmid_dw(self):
        pass

    def get_dxmid_dxin(self):
        pass

    def get_dxout_dxin(self):
        self.dxout_dxin = self.dxout_dxmid * self.dxmid_dxin

    def get_dxout_dw(self):
        self.dxout_dw = np.diag(self.dxout_dxmid) @ self.dxmid_dw

    def update_dloss_dw_dbias(self, dloss_dxout):
        self.dloss_dw += self.dxout_dw * dloss_dxout
        self.dloss_dbias += self.dxout_dbias * dloss_dxout

    def gradient_descent(self, eps):
        self.weights -= eps * self.dloss_dw
        self.bias -= eps * self.dloss_dbias

Se implementa una clase `Neuron`, que es la unidad básica de la red Kolmogorov-Arnold. Básicamente, la clase define una neurona como un objeto que puede recibir entradas, procesarlas con funciones no lineales internas y generar una salida. La configuración inicial de la neurona incluye definir el número de entradas (`n_in`), el número de pesos por conexión (`n_weights_per_edge`) y los valores iniciales de los pesos, que se generan aleatoriamente dentro de un rango. Cada neurona también tiene un sesgo (`bias`) que puede ajustarse durante el entrenamiento.

La estructura de la clase se centra en preparar todo lo necesario para implementar el paso hacia adelante (forward pass) y el cálculo de gradientes, que es crucial para el entrenamiento. Por ejemplo, al llamar a la neurona con una entrada (`__call__`), esta ejecuta el paso hacia adelante y calcula las derivadas parciales necesarias para ajustar los pesos y el sesgo según el error de la red. Varias funciones auxiliares, como `get_xmid`, `get_xout`, y `get_dxout_dxin`, son puntos clave que encapsulan diferentes etapas del procesamiento, pero no están implementadas todavía, ya que se espera que estas sean definidas más adelante en el tutorial.

Además, la neurona incluye mecanismos para realizar el descenso de gradiente, permitiendo ajustar sus parámetros según el error calculado. Esto se hace acumulando gradientes durante las iteraciones y luego actualizando los pesos y el sesgo proporcionalmente a estos gradientes y a la tasa de aprendizaje (`eps`). En resumen, esta clase establece la base para construir y entrenar una red Kolmogorov-Arnold, encapsulando tanto la funcionalidad de una neurona como su interacción con el resto de la red.

In [ ]:
import math

def relu(x, get_derivative=False):
    return x * (x > 0) if not get_derivative else 1.0 * (x >= 0)

def tanh_act(x, get_derivative=False):
    if not get_derivative:
        return math.tanh(x)
    return 1 - math.tanh(x) ** 2

def sigmoid_act(x, get_derivative=False):
    if not get_derivative:
        return 1 / (1 + math.exp(-x))
    return sigmoid_act(x) * (1 - sigmoid_act(x))

class NeuronNN(Neuron):

    def __init__(self, n_in, weights_range=None, activation=relu):
        super().__init__(n_in, n_weights_per_edge=1, weights_range=weights_range)
        self.activation = activation
        self.activation_input = None

    def get_xmid(self):
        self.xmid = self.weights[:, 0] * self.xin

    def get_xout(self):
        self.activation_input = sum(self.xmid.flatten()) + self.bias
        self.xout = self.activation(self.activation_input, get_derivative=False)

    def get_dxout_dxmid(self):
        self.dxout_dxmid = self.activation(self.activation_input, get_derivative=True) * np.ones(self.n_in)

    def get_dxout_dbias(self):
        self.dxout_dbias = self.activation(self.activation_input, get_derivative=True)

    def get_dxmid_dw(self):
        self.dxmid_dw = np.reshape(self.xin, (-1, 1))

    def get_dxmid_dxin(self):
        self.dxmid_dxin = self.weights.flatten()

En esta extensión de la clase Neuron, denominada NeuronNN, se implementa una neurona que utiliza funciones de activación para modelar relaciones no lineales entre las entradas y las salidas. Además de heredar la estructura básica de la clase Neuron, esta clase incorpora funcionalidades concretas para calcular las salidas intermedias y finales (xmid y xout) utilizando una función de activación configurable, como relu, tanh, o sigmoid.

La función de activación, que se pasa como parámetro al constructor, introduce no linealidades cruciales para el aprendizaje en redes neuronales. Dependiendo del contexto, estas funciones pueden calcular tanto el valor de activación como su derivada, lo cual es útil para el entrenamiento mediante backpropagation. Por ejemplo, relu(x) devuelve el valor rectificado de la entrada, mientras que su derivada es 1 para valores positivos y 0 para negativos.

El método get_xmid calcula los productos ponderados entre los valores de entrada y los pesos, almacenándolos como una representación intermedia (xmid). Posteriormente, el método get_xout suma estos valores junto con el sesgo y aplica la función de activación para obtener la salida final. La derivada de esta salida respecto a las entradas (get_dxout_dxmid) y respecto al sesgo (get_dxout_dbias) también se calcula utilizando la derivada de la función de activación seleccionada.

In [ ]:
from scipy.interpolate import BSpline

def get_bsplines(x_bounds, n_fun, degree=3, **kwargs):
    grid_len = n_fun - degree + 1
    step = (x_bounds[1] - x_bounds[0]) / (grid_len - 1)
    edge_fun, edge_fun_der = {}, {}

    edge_fun[0] = lambda x: x / (1 + np.exp(-x))
    edge_fun_der[0] = lambda x: (1 + np.exp(-x) + x * np.exp(-x)) / np.power((1 + np.exp(-x)), 2)

    t = np.linspace(x_bounds[0] - degree * step, x_bounds[1] + degree * step, grid_len + 2 * degree)
    t[degree], t[-degree - 1] = x_bounds[0], x_bounds[1]
    for ind_spline in range(n_fun - 1):
        edge_fun[ind_spline + 1] = BSpline.basis_element(t[ind_spline:ind_spline + degree + 2], extrapolate=False)
        edge_fun_der[ind_spline + 1] = edge_fun[ind_spline + 1].derivative()
    return edge_fun, edge_fun_der

class NeuronKAN(Neuron):

    def __init__(self, n_in, n_weights_per_edge, x_bounds, weights_range=None, get_edge_fun=get_bsplines, **kwargs):
        self.x_bounds = x_bounds
        super().__init__(n_in, n_weights_per_edge=n_weights_per_edge, weights_range=weights_range)
        self.edge_fun, self.edge_fun_der = get_edge_fun(self.x_bounds, self.n_weights_per_edge, **kwargs)

    def get_xmid(self):
        self.phi_x_mat = np.array([self.edge_fun[b](self.xin) for b in self.edge_fun]).T
        self.phi_x_mat[np.isnan(self.phi_x_mat)] = 0
        self.xmid = (self.weights * self.phi_x_mat).sum(axis=1)

    def get_xout(self):
        self.xout = tanh_act(sum(self.xmid.flatten()), get_derivative=False)

    def get_dxout_dxmid(self):
        self.dxout_dxmid = tanh_act(sum(self.xmid.flatten()), get_derivative=True) * np.ones(self.n_in)

    def get_dxmid_dw(self):
        self.dxmid_dw = self.phi_x_mat

    def get_dxmid_dxin(self):
        phi_x_der_mat = np.array([self.edge_fun_der[b](self.xin) if self.edge_fun[b](self.xin) is not None else 0
                                  for b in self.edge_fun_der]).T
        phi_x_der_mat[np.isnan(phi_x_der_mat)] = 0
        self.dxmid_dxin = (self.weights * phi_x_der_mat).sum(axis=1)

    def get_dxout_dbias(self):
        self.dxout_dbias = 0

Este código implementa la clase NeuronKAN, que extiende las funcionalidades de la clase Neuron para utilizar la arquitectura Kolmogorov-Arnold. En esta neurona, se incorporan funciones base de tipo B-spline, que son ideales para aproximar relaciones complejas debido a su flexibilidad y suavidad. Las funciones B-spline y sus derivadas se generan con la función get_bsplines, que define una serie de splines en el rango de las entradas (x_bounds), parametrizados según el número de funciones (n_fun) y su grado.

El método get_xout aplica una función no lineal (tanh) a la suma de las salidas intermedias, generando la salida final de la neurona. Por otro lado, los métodos get_dxmid_dw y get_dxmid_dxin calculan las derivadas necesarias para ajustar los pesos y modelar cómo los cambios en las entradas afectan las salidas intermedias. En particular, get_dxmid_dxin utiliza las derivadas de las funciones B-spline para evaluar cómo las variaciones en las entradas afectan a las respuestas de las funciones base.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_inputs = 3
inputs = np.array([0.5, -1.2, 0.8])
x_bounds = [-1, 1]
n_weights_per_edge = 4

neuron_nn = NeuronNN(n_in=n_inputs, weights_range=[-1, 1], activation=tanh_act)

neuron_kan = NeuronKAN(
    n_in=n_inputs,
    n_weights_per_edge=n_weights_per_edge,
    x_bounds=x_bounds,
    weights_range=[-1, 1]
)

output_nn = neuron_nn(inputs)
output_kan = neuron_kan(inputs)

print("Salida de la Neurona Clásica (NeuronNN):", output_nn)
print("Salida de la Neurona KAN (NeuronKAN):", output_kan)

edge_fun, _ = get_bsplines(x_bounds, n_weights_per_edge)
x_vals = np.linspace(-1.5, 1.5, 200)

plt.figure(figsize=(10, 6))
for i, func in edge_fun.items():
    y_vals = func(x_vals)
    plt.plot(x_vals, y_vals, label=f"Edge Function {i}")

plt.title("Funciones de Borde (Edge Functions) en Neurona KAN")
plt.xlabel("x")
plt.ylabel("Edge Function Output")
plt.legend()
plt.grid()
plt.show()

Se comparan las salidas de dos tipos de neuronas: una neurona clásica (NeuronNN) y una neurona Kolmogorov-Arnold (NeuronKAN), utilizando un conjunto de entradas definido por inputs. Primero, se inicializan ambas neuronas con parámetros específicos, como el rango de los pesos, la cantidad de entradas y, en el caso de la KAN, el rango de valores (x_bounds) y el número de funciones base (n_weights_per_edge). Luego, ambas neuronas calculan sus salidas para las entradas dadas.

La salida de la NeuronNN se obtiene aplicando una función de activación (tanh) a una combinación lineal de las entradas y los pesos. Por otro lado, la NeuronKAN utiliza funciones base B-spline para transformar las entradas antes de combinarlas con los pesos, permitiendo modelar relaciones más complejas entre entradas y salidas.

In [ ]:
class FullyConnectedLayer:

    def __init__(self, n_in, n_out, neuron_class=NeuronNN, **kwargs):
        self.n_in, self.n_out = n_in, n_out
        self.neurons = [neuron_class(n_in) if (kwargs == {}) else neuron_class(n_in, **kwargs) for _ in range(n_out)]
        self.xin = None
        self.xout = None
        self.dloss_dxin = None
        self.zero_grad()

    def __call__(self, xin):

        self.xin = xin
        self.xout = np.array([nn(self.xin) for nn in self.neurons])
        return self.xout

    def zero_grad(self, which=None):
        if which is None:
            which = ['xin', 'weights', 'bias']
        for w in which:
            if w == 'xin':
                self.dloss_dxin = np.zeros(self.n_in)
            elif w == 'weights':
                for nn in self.neurons:
                    nn.dloss_dw = np.zeros((self.n_in, self.neurons[0].n_weights_per_edge))
            elif w == 'bias':
                for nn in self.neurons:
                    nn.dloss_dbias = 0
            else:
                raise ValueError('input \'which\' value not recognized')

    def update_grad(self, dloss_dxout):
        for ii, dloss_dxout_tmp in enumerate(dloss_dxout):
            self.dloss_dxin += self.neurons[ii].dxout_dxin * dloss_dxout_tmp
            self.neurons[ii].update_dloss_dw_dbias(dloss_dxout_tmp)
        return self.dloss_dxin

n_inputs = 4
n_outputs = 3
inputs = np.array([0.5, -1.0, 0.8, 0.2])

layer_nn = FullyConnectedLayer(n_in=n_inputs, n_out=n_outputs, neuron_class=NeuronNN, weights_range=[-1, 1])


outputs = layer_nn(inputs)
print("Salidas de la capa con NeuronNN:", outputs)


layer_kan = FullyConnectedLayer(
    n_in=n_inputs,
    n_out=n_outputs,
    neuron_class=NeuronKAN,
    n_weights_per_edge=4,
    x_bounds=[-1, 1],
    weights_range=[-1, 1]
)

outputs_kan = layer_kan(inputs)
print("Salidas de la capa con NeuronKAN:", outputs_kan)

labels = [f"Neuron {i+1}" for i in range(n_outputs)]
plt.bar(labels, outputs, alpha=0.6, label="NeuronNN")
plt.bar(labels, outputs_kan, alpha=0.6, label="NeuronKAN")
plt.title("Salidas de NeuronNN y NeuronKAN en una capa")
plt.ylabel("Output Value")
plt.legend()
plt.grid()
plt.show()

El código define una capa completamente conectada que utiliza dos tipos de neuronas: unas clásicas y otras basadas en funciones B-spline, llamadas NeuronNN y NeuronKAN, respectivamente. La capa toma un conjunto de entradas, las procesa a través de las neuronas y devuelve sus salidas. La idea es comparar cómo cada tipo de neurona transforma las mismas entradas para entender sus diferencias. Las NeuronNN funcionan aplicando transformaciones lineales y funciones de activación tradicionales como tanh, mientras que las NeuronKAN utilizan funciones B-spline para capturar patrones más complejos en los datos. Luego, el gráfico de barras muestra las salidas de ambas capas para visualizar estas diferencias y ver cómo responden a las mismas entradas.

In [ ]:
class Loss:

    def __init__(self, n_in):
        self.n_in = n_in
        self.y, self.dloss_dy, self.loss, self.y_train = None, None, None, None

    def __call__(self, y, y_train):

        self.y, self.y_train = np.array(y), y_train
        self.get_loss()
        self.get_dloss_dy()
        return self.loss

    def get_loss(self):
        pass

    def get_dloss_dy(self):
        pass


class SquaredLoss(Loss):

    def get_loss(self):
        self.loss = np.mean(np.power(self.y - self.y_train, 2))

    def get_dloss_dy(self):
        self.dloss_dy = 2 * (self.y - self.y_train) / self.n_in


class CrossEntropyLoss(Loss):

    def get_loss(self):
        self.loss = - np.log(np.exp(self.y[self.y_train[0]]) / sum(np.exp(self.y)))

    def get_dloss_dy(self):
        self.dloss_dy = np.exp(self.y) / sum(np.exp(self.y))
        self.dloss_dy[self.y_train] -= 1

import numpy as np

n_in = 3
y_pred = np.array([0.5, 0.8, 0.2])
y_train = np.array([0.4, 1.0, 0.1])

squared_loss = SquaredLoss(n_in)
loss_value = squared_loss(y_pred, y_train)
gradient = squared_loss.dloss_dy

print("Squared Loss")
print("Pérdida:", loss_value)
print("Gradiente:", gradient)

y_pred_logits = np.array([2.0, 1.0, 0.1])
y_train_label = np.array([0])

cross_entropy_loss = CrossEntropyLoss(n_in)
loss_value_ce = cross_entropy_loss(y_pred_logits, y_train_label)
gradient_ce = cross_entropy_loss.dloss_dy

print("\nCross-Entropy Loss")
print("Pérdida:", loss_value_ce)
print("Gradiente:", gradient_ce)

El codigo anterior implementa dos funciones de pérdida comunes en aprendizaje automático: la pérdida cuadrática media (Squared Loss) y la pérdida de entropía cruzada (Cross-Entropy Loss). Estas funciones comparan las predicciones de un modelo con los valores verdaderos, y su propósito es medir qué tan bien o mal se está desempeñando el modelo. La pérdida cuadrática media se usa típicamente para tareas de regresión, calculando el promedio de los errores cuadrados entre las predicciones y los valores reales. Por otro lado, la pérdida de entropía cruzada está diseñada para problemas de clasificación, especialmente aquellos con salida como logits, evaluando la discrepancia entre la probabilidad predicha para la clase correcta y las probabilidades predichas totales.

In [ ]:
from tqdm import tqdm

class FeedForward:
    def __init__(self, layer_len, eps=.0001, seed=None, loss=SquaredLoss, **kwargs):
        self.seed = np.random.randint(int(1e4)) if seed is None else int(seed)
        np.random.seed(self.seed)
        self.layer_len = layer_len
        self.eps = eps
        self.n_layers = len(self.layer_len) - 1
        self.layers = [FullyConnectedLayer(layer_len[ii], layer_len[ii + 1], **kwargs) for ii in range(self.n_layers)]
        self.loss = loss(self.layer_len[-1])
        self.loss_hist = None

    def __call__(self, x):
        x_in = x
        for ll in range(self.n_layers):
            x_in = self.layers[ll](x_in)
        return x_in

    def backprop(self):

        delta = self.layers[-1].update_grad(self.loss.dloss_dy)
        for ll in range(self.n_layers - 1)[::-1]:
            delta = self.layers[ll].update_grad(delta)

    def gradient_descent_par(self):
        for ll in self.layers:
            for nn in ll.neurons:
                nn.gradient_descent(self.eps)

    def train(self, x_train, y_train, n_iter_max=10000, loss_tol=.1):
        self.loss_hist = np.zeros(n_iter_max)
        x_train, y_train = np.array(x_train), np.array(y_train)
        assert x_train.shape[0] == y_train.shape[0], 'x_train, y_train must contain the same number of samples'
        assert x_train.shape[1] == self.layer_len[0], 'shape of x_train is incompatible with first layer'

        pbar = tqdm(range(n_iter_max))
        for it in pbar:
            loss = 0
            for ii in range(x_train.shape[0]):
                x_out = self(x_train[ii, :])
                loss += self.loss(x_out, y_train[ii, :])
                self.backprop()
                [layer.zero_grad(which=['xin']) for layer in self.layers]
            self.loss_hist[it] = loss
            if (it % 10) == 0:
                pbar.set_postfix_str(f'loss: {loss:.3f}')
            if loss < loss_tol:
                pbar.set_postfix_str(f'loss: {loss:.3f}. Convergence has been attained!')
                self.loss_hist = self.loss_hist[: it]
                break
            self.gradient_descent_par()
            [layer.zero_grad(which=['weights', 'bias']) for layer in self.layers]

import numpy as np
import matplotlib.pyplot as plt

x_train = np.linspace(-1, 1, 50).reshape(-1, 1)
y_train = x_train ** 2

layer_len = [1, 10, 10, 1]


nn = FeedForward(layer_len=layer_len, eps=0.01, loss=SquaredLoss)

nn.train(x_train, y_train, n_iter_max=5000, loss_tol=0.01)

plt.plot(nn.loss_hist)
plt.title("Historial de Pérdida")
plt.xlabel("Iteraciones")
plt.ylabel("Pérdida")
plt.show()

x_test = np.linspace(-1.5, 1.5, 100).reshape(-1, 1)
y_pred = np.array([nn(xi) for xi in x_test]).reshape(-1)
s
plt.scatter(x_train, y_train, label="Datos de entrenamiento")
plt.plot(x_test, y_pred, color="red", label="Predicciones de la red")
plt.title("Aproximación de $y = x^2$")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

En el codigo anterior se implementa y entrena una red neuronal feedforward desde cero utilizando capas completamente conectadas y un esquema básico de retropropagación. Define cómo se construyen y conectan las capas, cómo se calculan las salidas de la red para un conjunto de entradas y cómo se ajustan los parámetros del modelo mediante el cálculo del gradiente y el descenso de gradiente. La red está configurada para minimizar la pérdida cuadrática (Squared Loss) entre las predicciones de la red y los valores reales, lo cual es esencial para problemas de regresión.

##Item II:
Obtenga resultados de ejecución de las ANN como en el tutorial. En el punto 5 del tutorial, modifique los datos de entrada usando otros datasets y vuelva a hacer comparaciones entre los resultados obtenidos de las distintas redes usadas en el tutorial.

###1D regression:

Para la regresión en una dimensión se comienza por establecer los X e Y de entrenamiento, la primera con 50 puntos uniformemente distribuidos en el intervalo [−1,1], mientras que Y se establece con una función no lineal generada con una onda sinusoidal sin(4x) y un decaimiento exponencial 𝑒 elevado a —(𝑥+1). Posteriormente, se grafican los datos obteniendo los datos reales.

In [ ]:
color_plots = {'dataset': 'b',
               'kan': 'orange',
               'mlp': 'green'}

x_train = np.linspace(-1, 1, 50).reshape(-1, 1)
y_train = .5 * np.sin(4 * x_train) * np.exp(-(x_train+1)) + .5  # damped sinusoid

fig, ax = plt.subplots(figsize=(4,3))
ax.plot(x_train, y_train, 'o', color=color_plots['dataset'])
ax.set_title('Training data')
ax.grid()
fig.tight_layout()

Para el entrenamiento de los modelos se establecen los parámetros en común como el número de iteraciones, el umbral para detener el entrenamiento si el error baja de cierto valor y la semilla.

In [ ]:
n_iter_train_1d = 500
loss_tol_1d = .05
seed = 476

La configuración de la red KAN se comienza por establecer los parámetros (1,2,2,1) los cuales determinan la entrada de 1 dimensión, 2 capas ocultas con 2 neuronas cada una y una salida.
Además, se establecen los hiperparámetros, tales como “eps” que corresponde a la tasa de aprendizaje, el tipo de neurona, el número de funciones base por conexión y el tipo de esta como B-splines.
Y posteriormente se ejecuta el entrenamiento con los datos configurados.

In [ ]:
kan_1d = FeedForward([1, 2, 2, 1],  # layer size
                  eps=.01,  # gradient descent parameter
                  n_weights_per_edge=7,  # n. edge functions
                  neuron_class=NeuronKAN,
                  x_bounds=[-1, 1],  # input domain bounds
                  get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                  seed=seed,
                  weights_range=[-1, 1])
kan_1d.train(x_train,
          y_train,
          n_iter_max=n_iter_train_1d,
          loss_tol=loss_tol_1d)

En el caso de MLP primero se establece su arquitectura, en este caso se tiene la entrada de 1 dimensión, una capa oculta con 13 neuronas y una salida.
Al igual que can se establecen los hiperparámetros como la tasa de aprendizaje (.005), la función de activación ReLU, el tipo de neurona. Y para finalizar se ejecuta el entrenamiento con los parámetros establecidos.

In [ ]:
mlp_1d = FeedForward([1, 13, 1],  # layer size
                  eps=.005,  # gradient descend parameter
                  activation=relu,  # activation type (ReLU, tanh or sigmoid)
                  neuron_class=NeuronNN,
                  seed=seed,
                  weights_range=[-.5, .5])
mlp_1d.train(x_train,
             y_train,
             n_iter_max=n_iter_train_1d,
             loss_tol=loss_tol_1d)

Para finalizar se grafican los resultados para poder apreciar y comparar las predicciones de KAN y MLP con los datos reales.

In [ ]:
fig, ax = plt.subplots(figsize=(4,3.2))
x_plot = np.linspace(x_train[0], x_train[-1], 1000).reshape(-1, 1)
ax.plot(x_train, y_train, 'o', color=color_plots['dataset'], label='training dataset')
ax.plot(x_plot, [kan_1d(x) for x in x_plot], color=color_plots['kan'], label='KAN')
ax.plot(x_train, [kan_1d(x) for x in x_train], 'x', color=color_plots['kan'], fillstyle='none')
ax.plot(x_plot, [mlp_1d(x) for x in x_plot], color=color_plots['mlp'], label='MLP')
ax.plot(x_train, [mlp_1d(x) for x in x_train], 'd', color=color_plots['mlp'], fillstyle='none')
ax.set_xlabel('input feature', fontsize=13)
ax.set_title('Regression', fontsize=15)
ax.legend()
ax.grid()
fig.tight_layout()
plt.show()

Además, se genera el gráfico de la evolución de la perdida, el cual nos permite observar el comportamiento de la función de perdida durante el entrenamiento. Este gráfico muestra como la perdida cambia a medida que avanza el entrenamiento.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_1d.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_1d.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_1d * np.ones(max(len(mlp_1d.loss_hist), len(kan_1d.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
fig1.tight_layout()
plt.show()

###1D regression II:

Para poder hacer comparaciones con los resultados se realizaron cambios en los parámetros de las distintas redes usadas con el fin de ver las variaciones que estos cambios provocan.

In [ ]:
x_train = np.linspace(-1, 1, 50).reshape(-1, 1)
y_train = .5 * np.sin(4 * x_train) * np.exp(-(x_train+1)) + .5  # damped sinusoid

fig, ax = plt.subplots(figsize=(4,3))
ax.plot(x_train, y_train, 'o', color=color_plots['dataset'])
ax.set_title('Training data')
ax.grid()
fig.tight_layout()

n_iter_train_1d = 500
loss_tol_1d = .05
seed = 476

kan_1d = FeedForward([1, 5, 7, 1],  # layer size
                  eps=.01,  # gradient descent parameter
                  n_weights_per_edge=7,  # n. edge functions
                  neuron_class=NeuronKAN,
                  x_bounds=[-1, 1],  # input domain bounds
                  get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                  seed=seed,
                  weights_range=[-1, 1])
kan_1d.train(x_train,
          y_train,
          n_iter_max=n_iter_train_1d,
          loss_tol=loss_tol_1d)

mlp_1d = FeedForward([1, 5, 1],  # layer size
                  eps=.005,  # gradient descend parameter
                  activation=relu,  # activation type (ReLU, tanh or sigmoid)
                  neuron_class=NeuronNN,
                  seed=seed,
                  weights_range=[-.5, .5])
mlp_1d.train(x_train,
             y_train,
             n_iter_max=n_iter_train_1d,
             loss_tol=loss_tol_1d)

fig, ax = plt.subplots(figsize=(4,3.2))
x_plot = np.linspace(x_train[0], x_train[-1], 1000).reshape(-1, 1)
ax.plot(x_train, y_train, 'o', color=color_plots['dataset'], label='training dataset')
ax.plot(x_plot, [kan_1d(x) for x in x_plot], color=color_plots['kan'], label='KAN')
ax.plot(x_train, [kan_1d(x) for x in x_train], 'x', color=color_plots['kan'], fillstyle='none')
ax.plot(x_plot, [mlp_1d(x) for x in x_plot], color=color_plots['mlp'], label='MLP')
ax.plot(x_train, [mlp_1d(x) for x in x_train], 'd', color=color_plots['mlp'], fillstyle='none')
ax.set_xlabel('input feature', fontsize=13)
ax.set_title('Regression', fontsize=15)
ax.legend()
ax.grid()
fig.tight_layout()
plt.show()

Las variaciones en las dimensiones de las redes neuronales generan notorios cambios en la gráfica que nos permiten sacar distintas conclusiones.
Al aumentar el número de capas y el número de neuronas, podemos notar cómo las oscilaciones en las gráficas de pérdida y precisión aumentan. Esto refleja una mayor sensibilidad del modelo a los datos y un mayor riesgo de sobreajuste (overfitting).

En el caso contrario, si se reducen demasiado las dimensiones del modelo (ya sea el número de capas o de neuronas), la capacidad del modelo para aprender patrones complejos disminuye significativamente, lo que resulta en un subajuste (underfitting).

En el caso específico de la red MLP, al disminuir el número de neuronas, las gráficas de pérdida y precisión tienden a mostrar curvas más rectas y estables. Esto ocurre porque la capacidad del modelo para aprender se limita, haciendo que los ajustes de los pesos sean más graduales y menos sensibles a pequeñas variaciones en los datos.

Aumentar las neuronas en una red MLP incrementa su capacidad de aprendizaje, permitiendo modelar patrones más complejos, pero también aumenta el riesgo de sobreajuste, el tiempo de entrenamiento y el consumo de recursos.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_1d.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_1d.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_1d * np.ones(max(len(mlp_1d.loss_hist), len(kan_1d.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
fig1.tight_layout()
plt.show()

###2D regression:

Se establece la función "fun2d" la cual genera un conjunto de datos bidimensionales, esta calcula el producto de la primera entrada y la raíz de la cuadrada de la segunda entrada. Posteriormente, se crean matrices para cada entrada usando "np.meshgrid" y se combinan en una matriz de entrenamiento "x_train2d".

In [ ]:
def fun2d(X1, X2):
    return X1 * np.power(X2, .5)

X1, X2 = np.meshgrid(np.linspace(0, .8, 8), np.linspace(0, 1, 10))
Y_training = fun2d(X1, X2)

x_train2d = np.concatenate((X1.reshape(-1, 1), X2.reshape(-1, 1)), axis=1)
y_train2d = Y_training.reshape(-1, 1)

f, a = plt.subplots(figsize=(3,3))
a.set_xlabel('input feature 1')
a.set_ylabel('input feature 2')
a.pcolor(X1, X2, Y_training)
a.set_title('Training data')
f.tight_layout()
plt.show()

n_iter_train_2d = 500
loss_tol_2d = .1
seed = 476

Luego se define una red neuronal KAN con una arquitectura que toma datos con dos características, una capa oculta con 2 neuronas y un único valor como salida.

Al igual que con una dimensión se establece la tasa de aprendizaje y se define el número de funciones base por conexión entre neuronas, esto determina la flexibilidad de la red, además se determina el tipo de neurona, los límites del dominio y el tipo de funciones base. Y finaliza con el entrenamiento con los parámetros antes mencionados.

In [ ]:
kan_2d = FeedForward([2, 2, 1],  # layer size
                  eps=.02,  # gradient descent parameter
                  n_weights_per_edge=10,  # n. edge functions
                  neuron_class=NeuronKAN,
                  x_bounds=[-1, 1],  # input domain bounds
                  get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                  seed=seed,
                  weights_range=[-1, 1])
kan_2d.train(x_train2d,
          y_train2d,
          n_iter_max=n_iter_train_2d,
          loss_tol=loss_tol_2d)

Luego se establece la red neuronal de tipo MLP con una arquitectura de 2 entradas, 6 neuronas en la capa oculta y una unica salida, se establece la funcion de activacion ReLU, y al igual que todas las otras configuraciones se establece un rango de pesos entre -0.1 y 0.1.
Por ultimo se ejecuta el aprendizaje con los datos configurados.

In [ ]:
mlp_2d = FeedForward([2, 6, 1],  # layer size
                     eps=.0025,  # gradient descend parameter
                     activation=relu,  # activation type (ReLU, tanh or sigmoid)
                     neuron_class=NeuronNN,
                     seed=seed,
                     weights_range=[-.1, .1])
mlp_2d.train(x_train2d,
             y_train2d,
             n_iter_max=n_iter_train_2d,
             loss_tol=loss_tol_2d)

Y_kan = np.array([kan_2d(xx) for xx in x_train2d])
Y_mlp = np.array([mlp_2d(xx) for xx in x_train2d])

f, a = plt.subplots(1, 3, figsize=(11,3))

vmin, vmax = min(Y_training.flatten()), max(Y_training.flatten())

im0 = a[0].pcolor(X1, X2, Y_training, vmin=vmin, vmax=vmax)
f.colorbar(im0, ax=a[0])
a[0].set_title('Training data')
a[0].set_xlabel('input feature 1')
a[0].set_ylabel('input feature 2')

im1 = a[1].pcolor(X1, X2, Y_kan.reshape(X1.shape), vmin=vmin, vmax=vmax)
f.colorbar(im1, ax=a[1])
a[1].set_title('KAN regression')
a[1].set_xlabel('input feature 1')
a[1].set_ylabel('input feature 2')

im2 = a[2].pcolor(X1, X2, Y_mlp.reshape(X1.shape), vmin=vmin, vmax=vmax)
f.colorbar(im2, ax=a[2])
a[2].set_title('MLP regression')
a[2].set_xlabel('input feature 1')
a[2].set_ylabel('input feature 2')

f.tight_layout()
plt.show()
# f.savefig(folder_fig + 'regr2D.png', dpi=500)

fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_2d.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_2d.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_2d * np.ones(max(len(mlp_2d.loss_hist), len(kan_2d.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('squared loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
plt.show()

###2D regression II

Al igual que en una dimencion al aumentar las neuronas se mejora el ajuste a datos complejos pero puede causar sobrajuste y mayor tiempo de entrenamiento.
En contraste al disminuir el numero de neuronas se pierde la capacidad de aprender patrones complejos, resultando en un subajuste.

In [ ]:
def fun2d(X1, X2):
    return X1 * np.power(X2, .5)

X1, X2 = np.meshgrid(np.linspace(0, .8, 8), np.linspace(0, 1, 10))
Y_training = fun2d(X1, X2)

x_train2d = np.concatenate((X1.reshape(-1, 1), X2.reshape(-1, 1)), axis=1)
y_train2d = Y_training.reshape(-1, 1)

f, a = plt.subplots(figsize=(3,3))
a.set_xlabel('input feature 1')
a.set_ylabel('input feature 2')
a.pcolor(X1, X2, Y_training)
a.set_title('Training data')
f.tight_layout()
plt.show()

n_iter_train_2d = 500
loss_tol_2d = .1
seed = 476

kan_2d = FeedForward([2, 5, 1],  # layer size
                  eps=.02,  # gradient descent parameter
                  n_weights_per_edge=10,  # n. edge functions
                  neuron_class=NeuronKAN,
                  x_bounds=[-1, 1],  # input domain bounds
                  get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                  seed=seed,
                  weights_range=[-1, 1])
kan_2d.train(x_train2d,
          y_train2d,
          n_iter_max=n_iter_train_2d,
          loss_tol=loss_tol_2d)

mlp_2d = FeedForward([2, 1, 1],  # layer size
                     eps=.0025,  # gradient descend parameter
                     activation=relu,  # activation type (ReLU, tanh or sigmoid)
                     neuron_class=NeuronNN,
                     seed=seed,
                     weights_range=[-.1, .1])
mlp_2d.train(x_train2d,
             y_train2d,
             n_iter_max=n_iter_train_2d,
             loss_tol=loss_tol_2d)

Y_kan = np.array([kan_2d(xx) for xx in x_train2d])
Y_mlp = np.array([mlp_2d(xx) for xx in x_train2d])

f, a = plt.subplots(1, 3, figsize=(11,3))

vmin, vmax = min(Y_training.flatten()), max(Y_training.flatten())

im0 = a[0].pcolor(X1, X2, Y_training, vmin=vmin, vmax=vmax)
f.colorbar(im0, ax=a[0])
a[0].set_title('Training data')
a[0].set_xlabel('input feature 1')
a[0].set_ylabel('input feature 2')

im1 = a[1].pcolor(X1, X2, Y_kan.reshape(X1.shape), vmin=vmin, vmax=vmax)
f.colorbar(im1, ax=a[1])
a[1].set_title('KAN regression')
a[1].set_xlabel('input feature 1')
a[1].set_ylabel('input feature 2')

im2 = a[2].pcolor(X1, X2, Y_mlp.reshape(X1.shape), vmin=vmin, vmax=vmax)
f.colorbar(im2, ax=a[2])
a[2].set_title('MLP regression')
a[2].set_xlabel('input feature 1')
a[2].set_ylabel('input feature 2')

f.tight_layout()
plt.show()
# f.savefig(folder_fig + 'regr2D.png', dpi=500)

Podemos apreciar que el aumento de neuronas en la red KAN produce un desorden en los parámetros lo que denota un sobreajuste lo que provoca que no generalice bien a nuevos datos.
Por otro lado, podemos apreciar que la esquina más oscura se produce por el bajo número de neuronas lo que provoca que el modelo no tenga suficiente capacidad para aprender patrones complejos en la esquina inferior.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_2d.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_2d.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_2d * np.ones(max(len(mlp_2d.loss_hist), len(kan_2d.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('squared loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
plt.show()

En el gráfico de evolución de perdida podemos apreciar como la línea de la red KAN muy oscilante y errática, esto gracias a que se presenta mayor complejidad en el modelo con el aumento de neuronas, provocando que la red comience a ajustar los parámetros de manera excesivamente fina.
Esto contrasta con la línea de la red MLP la cual se presenta más recta y sin perturbaciones fuertes, esto gracias a que al usar una sola neurona se tiene una capacidad limitada de aprendizaje y no permite la flexibilidad para adaptarse a los detalles finos.

###Classification I:

En esta parte se utilizan los datasets de sklearn.
En el primer codigo extraido del tutorial, se establece el dataset a utlizar "make_moons", se establece el conjunto de muestras en 50 y el ruido en 0.1, luego se normalizan los datos para que los valores esten en el rango de -1 a 1 para finalizar observando los datos en un grafico de dispersion.

In [ ]:
from sklearn import datasets

n_samples = 50
noise = 0.1
x_train_cl, y_train_cl = datasets.make_moons(n_samples=n_samples, noise=noise)
# normalize between -1 and 1
x_train_cl[:, 0] = (x_train_cl[:, 0] - min(x_train_cl[:, 0])) / max(x_train_cl[:, 0] - min(x_train_cl[:, 0])) * 2 - 1
x_train_cl[:, 1] = (x_train_cl[:, 1] - min(x_train_cl[:, 1])) / max(x_train_cl[:, 1] - min(x_train_cl[:, 1])) * 2 - 1

fig, ax = plt.subplots(figsize=(4,3))
ax.scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax.set_xlabel('input feature 1')
ax.set_ylabel('input feature 2')
ax.grid()
ax.set_title("Training data")
plt.show()

n_iter_train_cl = int(1e3)
loss_tol_cl = 1.5
seed = 476

En esta red tipo KAN para un problema de clasificación se establecen las dimensiones de la red en 2 capas con 2 neuronas cada una, se establecen los parámetros al igual que en las demás redes, estableciendo el tipo de neurona, la función de perdida el rango de entrada y la semilla para posteriormente ejecutar el entrenamiento con los datos antes mencionados.

In [ ]:
kan_cl = FeedForward([2, 2],  # layer size
                     eps=.02,  # gradient descent parameter
                     n_weights_per_edge=8,  # n. edge functions
                     neuron_class=NeuronKAN,
                     loss=CrossEntropyLoss,
                     x_bounds=[-1, 1],  # input domain bounds
                     get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                     seed=seed,
                     weights_range=[-1, 1])
kan_cl.train(x_train_cl,
             y_train_cl.reshape(-1, 1),
             n_iter_max=n_iter_train_cl,
             loss_tol=loss_tol_cl)

Para entrenar una red neuronal MLP para clasificación se establecen las dimensiones de esta, las cuales son 4 capas, la primera con 2 neuronas, las siguientes 2 capas con 10 neuronas cada una y la de salida con 2 neuronas también. Se establece la función de activación "tanh_act" y el tipo de neurona para finalmente ejecutar el entrenamiento con los datos configurados de la siguiente forma.

In [ ]:
mlp_cl = FeedForward([2, 10, 10, 2],  # layer size
                     eps=.01,  # gradient descend parameter
                     activation=tanh_act,  # activation type (relu, tanh_act or sigmoid_act)
                     neuron_class=NeuronNN,
                     loss=CrossEntropyLoss,
                     seed=seed,
                     weights_range=[-1, 1])
mlp_cl.train(x_train_cl,
             y_train_cl.reshape(-1, 1),
             n_iter_max=n_iter_train_cl,
             loss_tol=loss_tol_cl)

def softmax(vec):
    return np.exp(vec) / sum(np.exp(vec))

X1_cl, X2_cl = np.meshgrid(np.linspace(-1, 1, 40), np.linspace(-1, 1, 50))
x_cl = np.concatenate((X1_cl.reshape(-1, 1), X2_cl.reshape(-1, 1)), axis=1)

Y_kan_cl = np.array([softmax(kan_cl(x))[1] for x in x_cl]).reshape(X1_cl.shape)
Y_mlp_cl = np.array([softmax(mlp_cl(x))[1] for x in x_cl]).reshape(X1_cl.shape)

fig_cl, ax_cl = plt.subplots(1, 3, figsize=(12,3))
# vmin, vmax = min(Y_training.flatten()), max(Y_training.flatten())

ax_cl[0].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[0].grid()
ax_cl[0].set_title('Training data')
ax_cl[0].set_xlabel('input feature 1')
ax_cl[0].set_ylabel('input feature 2')

im0 = ax_cl[1].pcolor(X1_cl, X2_cl, Y_kan_cl, vmin=0, vmax=1, cmap=plt.cm.bwr)
ax_cl[1].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[1].set_title('KAN classification')
ax_cl[1].set_xlabel('input feature 1')
ax_cl[1].set_ylabel('input feature 2')

im1 = ax_cl[2].pcolor(X1_cl, X2_cl, Y_mlp_cl, vmin=0, vmax=1, cmap=plt.cm.bwr)
ax_cl[2].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[2].set_title('MLP classification')
ax_cl[2].set_xlabel('input feature 1')
ax_cl[2].set_ylabel('input feature 2')

fig_cl.colorbar(im1, ax=ax_cl[0])
fig_cl.colorbar(im1, ax=ax_cl[1])
fig_cl.colorbar(im1, ax=ax_cl[2])
fig_cl.tight_layout()
plt.show()
# fig_cl.savefig(folder_fig + 'classification.png', dpi=500)

En los gráficos podemos apreciar la clasificación de las redes neuronales antes configuradas y entrenadas, podemos ver ambos grupos, rojo y azul y su línea de separación que corresponde a la frontera de decisión, esta línea que separa las dos clases es el lugar donde el modelo cambia, podemos notar que en la red KAN esta línea es mucho más marcada y las clases están claramente separadas, se puede decir que ambos modelos aprendieron de manera efectiva.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_cl.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_cl.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_cl * np.ones(max(len(mlp_cl.loss_hist), len(kan_cl.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('cross-entropy loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
fig1.tight_layout()
plt.show()

###Classification II:

En esta segunda red de clasificación se realizaron cambios solo en el dataset utilizado y no en las dimensiones de la red.

In [ ]:
from sklearn import datasets

n_samples = 50
noise = 0.1
x_train_cl, y_train_cl = datasets.make_circles(n_samples=n_samples, noise=noise)
# normalize between -1 and 1
x_train_cl[:, 0] = (x_train_cl[:, 0] - min(x_train_cl[:, 0])) / max(x_train_cl[:, 0] - min(x_train_cl[:, 0])) * 2 - 1
x_train_cl[:, 1] = (x_train_cl[:, 1] - min(x_train_cl[:, 1])) / max(x_train_cl[:, 1] - min(x_train_cl[:, 1])) * 2 - 1

fig, ax = plt.subplots(figsize=(4,3))
ax.scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax.set_xlabel('input feature 1')
ax.set_ylabel('input feature 2')
ax.grid()
ax.set_title("Training data")
plt.show()

en este caso se utilizó también un dataset de sklearn, la función llamada "make_circles" genera un conjunto de datos en forma de círculos.

In [ ]:
n_iter_train_cl = int(1e3)
loss_tol_cl = 1.5
seed = 476

kan_cl = FeedForward([2, 2],  # layer size
                     eps=.02,  # gradient descent parameter
                     n_weights_per_edge=8,  # n. edge functions
                     neuron_class=NeuronKAN,
                     loss=CrossEntropyLoss,
                     x_bounds=[-1, 1],  # input domain bounds
                     get_edge_fun=get_bsplines,  # edge function type (B-splines ot Chebyshev)
                     seed=seed,
                     weights_range=[-1, 1])
kan_cl.train(x_train_cl,
             y_train_cl.reshape(-1, 1),
             n_iter_max=n_iter_train_cl,
             loss_tol=loss_tol_cl)

mlp_cl = FeedForward([2, 10, 10, 2],  # layer size
                     eps=.01,  # gradient descend parameter
                     activation=tanh_act,  # activation type (relu, tanh_act or sigmoid_act)
                     neuron_class=NeuronNN,
                     loss=CrossEntropyLoss,
                     seed=seed,
                     weights_range=[-1, 1])
mlp_cl.train(x_train_cl,
             y_train_cl.reshape(-1, 1),
             n_iter_max=n_iter_train_cl,
             loss_tol=loss_tol_cl)

def softmax(vec):
    return np.exp(vec) / sum(np.exp(vec))

X1_cl, X2_cl = np.meshgrid(np.linspace(-1, 1, 40), np.linspace(-1, 1, 50))
x_cl = np.concatenate((X1_cl.reshape(-1, 1), X2_cl.reshape(-1, 1)), axis=1)

Y_kan_cl = np.array([softmax(kan_cl(x))[1] for x in x_cl]).reshape(X1_cl.shape)
Y_mlp_cl = np.array([softmax(mlp_cl(x))[1] for x in x_cl]).reshape(X1_cl.shape)

fig_cl, ax_cl = plt.subplots(1, 3, figsize=(12,3))
# vmin, vmax = min(Y_training.flatten()), max(Y_training.flatten())

ax_cl[0].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[0].grid()
ax_cl[0].set_title('Training data')
ax_cl[0].set_xlabel('input feature 1')
ax_cl[0].set_ylabel('input feature 2')

im0 = ax_cl[1].pcolor(X1_cl, X2_cl, Y_kan_cl, vmin=0, vmax=1, cmap=plt.cm.bwr)
ax_cl[1].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[1].set_title('KAN classification')
ax_cl[1].set_xlabel('input feature 1')
ax_cl[1].set_ylabel('input feature 2')

im1 = ax_cl[2].pcolor(X1_cl, X2_cl, Y_mlp_cl, vmin=0, vmax=1, cmap=plt.cm.bwr)
ax_cl[2].scatter(*x_train_cl.T, c=y_train_cl, cmap=plt.cm.bwr)
ax_cl[2].set_title('MLP classification')
ax_cl[2].set_xlabel('input feature 1')
ax_cl[2].set_ylabel('input feature 2')

fig_cl.colorbar(im1, ax=ax_cl[0])
fig_cl.colorbar(im1, ax=ax_cl[1])
fig_cl.colorbar(im1, ax=ax_cl[2])
fig_cl.tight_layout()
plt.show()
# fig_cl.savefig(folder_fig + 'classification.png', dpi=500)

Como podemos apreciar en estos graficos aunque en la red KAN se puede apreciar una linea blanca precisa aun asi podemos observar muchos puntos rojos y azules sobre ella lo que denota que el modelo esta luchando por realizar una correcta separacion de clases, esto se puede deber a que el modelo no es el adecuado para los datos.

En la red MLP podemos ver aun mas desorden y una frontera de decision poco precisa, esto se debe al sobreajuste al tener 2 capas con 10 neruonas cada una, lo que hace que no pueda aprender adecuadamente la sepracion entre clases.

In [ ]:
color_plots = {'dataset': 'b',
               'kan': 'red',
               'mlp': 'purple'}

fig1, ax1 = plt.subplots(figsize=(4,3))
ax1.plot(kan_cl.loss_hist, '-', color=color_plots['kan'], label='KAN')
ax1.plot(mlp_cl.loss_hist, '-', color=color_plots['mlp'], label='MLP')
ax1.plot(loss_tol_cl * np.ones(max(len(mlp_cl.loss_hist), len(kan_cl.loss_hist))), '--k')
ax1.grid()
ax1.set_ylabel('cross-entropy loss')
ax1.set_xlabel('iteration', fontsize=13)
ax1.legend()
ax1.set_title('Loss evolution')
fig1.tight_layout()
plt.show()

##Item 3:
Explique en sus propias palabras cómo está construida la arquitectura KAN y por qué funciona. Además, haga un análisis sobre los resultados obtenidos y cómo y por qué podrían variar.

Esta innovadora arquitectura está construida basándose en una representación universal de funciones multivariables propuesta por los matemáticos Kolmogorov y Arnold. Estas redes reciben varias variables de entrada que corresponden a las características del problema y en sus capas internas se presentan las funciones de activación no lineales las cuales permiten aprender representaciones no lineales de los datos lo que ayuda para captar relaciones complejas, estas funciones se aplican también en las salidas con el fin de que se combinen para formar una representación cada vez más complejizada de la entrada.
Este tipo de funciones hacen que no se necesiten muchas capas para entrenar bien a la red lo que la hace más eficiente que otros tipos de red.

La arquitectura KAN (Kernels Associated with Neurons) se basa en una idea clave: en lugar de usar pesos fijos para cada conexión entre las capas de una red neuronal, asigna múltiples pesos por conexión que son modulados por funciones base como los B-splines. Esto permite que las conexiones no solo sean valores estáticos, sino que cambien dinámicamente según el valor de entrada. Así, cada entrada influye de mejor manera en las salidas de la red, lo que incrementa la capacidad de modelar relaciones complejas y no lineales. En los códigos que se trabajan, además se vió cómo estas funciones base, junto con los pesos, crean un sistema más flexible para calcular las salidas de cada neurona. Esto contrasta con las redes tradicionales, que simplemente multiplican entradas y pesos antes de aplicar una función de activación.

En cuanto a los resultados, al comparar la arquitectura KAN con una red clásica (NeuronNN), se ve que KAN ofrece mayor capacidad para aproximar funciones complejas. Por ejemplo, en el caso de aproximar
𝑦
=
𝑥
2

 , KAN podría modelar mejor los valores intermedios o cambios sutiles en la curva porque sus pesos por conexión permiten mayor adaptabilidad. Sin embargo, esto depende mucho de la cantidad de funciones base utilizadas y de cómo están distribuidos los datos de entrada. Por ejemplo, con datos muy dispersos o insuficientes, KAN podría no ajustarse correctamente, ya que tiene más parámetros que optimizar y podría sobreajustarse.